If you're opening this Notebook on colab, you will probably need to install 🤗 Transformers and 🤗 Datasets. Uncomment the following cell and run it.

In [ ]:
#! pip install datasets transformers
#!pip install datasets==4.8.4
#!pip install transformers
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 10.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


If you're opening this notebook locally, make sure your environment has an install from the last version of those libraries.

In [ ]:
#from huggingface_hub import notebook_login

#notebook_login()

In [ ]:
#!apt install git-lfs
!pip install torch

In [ ]:
!pip install av
!pip install torchcodec
!pip install av torchcodec
!pip install torchvision

# 1. Install specific stable PyAV version and Torchcodec
#!pip install av==11.0.0 torchcodec datasets accelerate --upgrade

# 2. Force an update to the torchvision package if backend integration is stuck
#!pip install torchvision --upgrade --no-cache-dir

In [ ]:
#!pip install --force-reinstall torchaudio --index-url https://pytorch.org
#!pip uninstall torch torchvision torchaudio -y
#!pip install torch torchvision torchaudio --index-url https://pytorch.org

In [1]:
from torchcodec.decoders import VideoDecoder
import sys
import torchvision.io

# Create a dummy object to satisfy the internal import check
class DummyVideoReader:
    pass

torchvision.io.VideoReader = DummyVideoReader
sys.modules['torchvision.io'].VideoReader = DummyVideoReader


running above block needed to successfully import videoDecoder in next block(avoids:- "Failed: cannot import name 'VideoReader' from 'torchvision.io' (/usr/local/lib/python3.12/dist-packages/torchvision/io/__init__.py)" error)

In [2]:
import os
# Force torchvision to look for the PyAV backend explicitly
os.environ["TORCHVISION_VIDEO_BACKEND"] = "pyav"

import torch
import torchvision
from datasets import Dataset

# Verify the class is now safely bound into the namespace
try:
    from torchvision.io import VideoReader
    print("Success: VideoReader successfully imported!")
except ImportError as e:
    print(f"Failed: {e}")

Success: VideoReader successfully imported!


Make sure your version of Transformers is at least 4.11.0 since the functionality was introduced in that version:

In [ ]:
import transformers

print(transformers.__version__)

5.13.1


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import os

print(os.getcwd())

path ="/content/gdrive/MyDrive/"
print(os.listdir(path))

/content
['analytics_vidhya', 'BERT_Model', 'Colab Notebooks', 'creditvidya_model_folder', 'Talentica_ML', 'weather_forcast', 'zenatic_ac_model_folders', 'AC_Data_hour_with_temp1.xlsx', 'AC_Data_hour_with_temp.xlsx', 'AC_Data_hour.xlsx', 'AC_Data.csv', 'hiring_assignment_cv.xlsx', 'python_intro.ipynb', 'stock_forcast_model.h5', 'test_new_6thsense.csv', 'ner_dataset.csv', 'Large_videos_pics_originaldrive', 'upsc_2026_speciesInNews.docx', 'Large_videos_part2', 'distilgpt2-finetuned-wikitext2']


You can find a script version of this notebook to fine-tune your model in a distributed fashion using multiple GPUs or TPUs [here](https://github.com/huggingface/transformers/tree/master/examples/language-modeling).

# Fine-tuning a language model

In this notebook, we'll see how to fine-tune 🤗 autoregressive/causal transformers based model for NLP. We'll train mode by **trainer wrapper** as well as **accelerator loop(latter for more customization)**. So what are causal models?

- Causal language modeling: These are **class of decoder based(generative model) model**. The model has to predict the next token in the sentence (so the labels are the same as the inputs shifted to the right). To make sure the model does not cheat, it gets an ***attention mask that will prevent it to access the tokens after token i when trying to predict the token i+1 in the sentence.***

- This masking property and ability to generate next words(without predifined output labels) is ***basis of modern LLMs(GPT series) and Generative-AI***.



## Preparing the dataset

For each of those tasks, we will use the [Wikitext 2]() dataset as an example. You can load it very easily with the 🤗 Datasets library.

In [ ]:
from datasets import load_dataset
datasets = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1')

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

You can replace the dataset above with any dataset hosted on [the hub](https://huggingface.co/datasets) (🤗) or use your own files. Just uncomment the following cell and replace the paths with values that will lead to your files:

In [ ]:
# datasets = load_dataset("text", data_files={"train": path_to_train.txt, "validation": path_to_validation.txt}

You can also load datasets from a csv or a JSON file, see the [full documentation](https://huggingface.co/docs/datasets/loading_datasets.html#from-local-files) for more information.

To access an actual element, you need to select a split first, then give an index:

In [ ]:
datasets["train"][1]

{'text': ' = Valkyria Chronicles III = \n'}

In [ ]:
type(datasets), datasets.keys(), type(datasets["train"]), datasets["train"][10].keys()

(datasets.dataset_dict.DatasetDict,
 dict_keys(['test', 'train', 'validation']),
 datasets.arrow_dataset.Dataset,
 dict_keys(['text']))

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset.

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [ ]:
show_random_elements(datasets["train"])

,text
0,but most she loathed the hour \n
1,= = Personnel = = \n
2,"Having previously worked on D.R. & Quinch for 2000 AD , a title made popular by John Constantine 's creator Alan Moore , Delano was selected to start the character 's first run in his own comic by then editor Karen Berger in 1988 . Delano 's run was characterised by his political satire , taking on late 1980s and 1990s tropes such as with city financiers being literal demons , and Constantine meeting with Freemasons from the Houses of Parliament . He also had environmentalist issues crop up , especially in "" The Fear Machine "" ( issues # 15 @-@ 22 ) , where John fell in with a travelling community of environmental activists . Indeed , editor Karen Berger noted on Delano 's departure the irony that his final issue was handed in the week that Margaret Thatcher was forced out of office . \n"
3,"The helmet and visor were found in May 2010 in pastureland on a farm owned by Eric Robinson at Crosby Garrett in Cumbria . The finder , an unnamed metal detectorist in his 20s from Peterlee , County Durham , had been detecting with his father in two adjacent fields for some years but had previously only discovered some Roman coins and other small artefacts . The findspot is situated not far from a Roman road . A number of earthworks are located nearby , indicating the presence of a previously unrecorded ancient settlement . The area was strategically placed on the route to the northern frontier of Roman Britain within the territory of the Carvetii tribe . The Roman army would have been present in the area and would certainly have used the nearby road . A Roman auxiliary fort stood only 9 kilometres ( 5 @.@ 6 mi ) to the north @-@ east at Verterae ( Brough Castle ) . \n"
4,
5,"The Kyra character has received mixed feedback from critics , and has been defined by her sex appeal and called "" overtly sexual , coy and kitteny "" , and "" tasty "" . Critics have positively and negatively compared the role to Carpenter 's previous performance as Cordelia Chase on the supernatural dramas Buffy the Vampire Slayer ( 1997 ) and Angel ( 1999 ) ; Demian of Television Without Pity criticized the character as a copy of Chase because of her lack of a unique identity . \n"
6,"HBO said its marketing plan for the series was , "" its largest , most aggressive push for a new series "" . The channel broadcast the first three episodes seven days a week at various times during the day . Non @-@ subscribers could preview the first two episodes during the first week of September 2005 . HBO implemented an outdoor marketing campaign in major cities and produced movie @-@ style trailers which preceded a number of films in cinemas . Entertainment Weekly , Vanity Fair , Time , and GQ published full @-@ size articles about the series . The History Channel broadcast five nights of documentaries featuring the Roman Empire , which were hosted by Stevenson , McKidd , and Varma , a collaboration which was the first of its kind between the two networks . \n"
7,
8,"Drought conditions between 1175 and 1180 prompted the Crusaders to sign a two @-@ year truce with the Muslims , but without Tripoli included in the terms . During the 1180s raids by Christians and Muslims into each other 's territory became more frequent . In 1180 , Saladin ventured into the County of Tripoli , ravaging the area . Unwilling to meet him in open battle , the Crusaders retreated to the relative safety of their fortifications . Without capturing the castles , Saladin could not secure control of the area , and once he retreated the Hospitallers were able to revitalize their damaged lands . The Battle of Hattin in 1187 was a disastrous defeat for the Crusaders : Guy of Lusignan , King of Jerusalem , was captured , as was the True Cross , a relic discovered during the First Crusade . Afterwards Saladin ordered the execution of the captured Templar and Hospitaller knights , such was the importance of the two orders in defending the Crusader

As we can see, some of the texts are a full paragraph of a Wikipedia article while others are just titles or empty lines.

## Causal Language modeling

For causal language modeling (CLM) we are going to take all the texts in our dataset and concatenate them after they are tokenized. Then we will split them in examples of a certain sequence length. This way the model will receive chunks of contiguous text that may look like:
```
part of text 1
```
or
```
end of text 1 [BOS_TOKEN] beginning of text 2
```
depending on whether they span over several of the original texts in the dataset or not. The labels will be the same as the inputs, shifted to the left.

We will use the [`distilgpt2`](https://huggingface.co/distilgpt2) model for this example. You can pick any of the checkpoints listed [here](https://huggingface.co/models?filter=causal-lm) instead:

In [ ]:
model_checkpoint = "distilgpt2"
#model_checkpoint ="distilbert/distilgpt2"

To tokenize all our texts with the same vocabulary that was used when training the model, we have to download a pretrained tokenizer. This is all done by the `AutoTokenizer` class:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

We can now call the tokenizer on all our texts. This is very simple, using the [`map`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasets.Dataset.map) method from the Datasets library. First we define a function that call the tokenizer on our texts:

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

Then we apply it to all the splits in our `datasets` object, using `batched=True` and 4 processes to speed up the preprocessing. We won't need the `text` column afterward, so we discard it.

In [ ]:
# Check if pad_token is already assigned; if not, set it to the eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Above block to only be used where uneven individual train blocks/data. Since we'll batch of equal non-padded blocks, using the above code redundant

In [ ]:
tokenized_datasets = datasets.map(tokenize_function, batched=True, num_proc=8, remove_columns=["text"])#default batch size =1000, means processing in chunks of 1000

Map (num_proc=8):   0%|          | 0/4358 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/36718 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/3760 [00:00<?, ? examples/s]

If we now look at an element of our datasets, we will see the text have been replaced by the `input_ids` the model will need:

In [ ]:
print(tokenized_datasets["train"][1])
print(type(tokenized_datasets["train"]))

{'input_ids': [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}
<class 'datasets.arrow_dataset.Dataset'>


In [ ]:
tokenizer.decode(tokenized_datasets["train"][1])

' = Valkyria Chronicles III = \n'

Now for the harder part: we need to concatenate all our texts together then split the result in small chunks of a certain `block_size`. To do this, we will use the `map` method again, with the option `batched=True`. This option actually lets us change the number of examples in the datasets by returning a different number of examples than we got. This way, we can create our new samples from a batch of examples.

First, we grab the maximum length our model was pretrained with. This might be a big too big to fit in your GPU RAM, so here we take a bit less at just 128.

In [ ]:
# block_size = tokenizer.model_max_length
block_size = 128

Then we write the preprocessing function that will group our texts:

In [ ]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
        # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

First note that we duplicate the inputs for our labels. This is because the model of the 🤗 Transformers library apply the shifting to the right, so we don't need to do it manually.

Also note that by default, the `map` method will send a batch of 1,000 examples to be treated by the preprocessing function. So here, we will drop the remainder to make the concatenated tokenized texts a multiple of `block_size` every 1,000 examples. You can adjust this behavior by passing a higher batch size (which will also be processed slower). You can also speed-up the preprocessing by using multiprocessing:

In [ ]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,#default batch_size
    num_proc=8,
)# here batch inside map only facilitates faster processing, does not group final data in batches like pytorch dataloader(so keep batch size =32, 64 in dataloader)

Map (num_proc=8):   0%|          | 0/4358 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/36718 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
print(type(lm_datasets["train"]), type(lm_datasets["validation"]))#not list of dict

<class 'datasets.arrow_dataset.Dataset'> <class 'datasets.arrow_dataset.Dataset'>


And we can check our datasets have changed: now the samples contain chunks of `block_size` contiguous tokens, potentially spanning over several of our original texts.

In [ ]:
tokenizer.decode(lm_datasets["train"][1]["input_ids"])

' game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Oz'

In [ ]:
tokenizer.decode(lm_datasets["train"][1]["labels"])

' game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Oz'

In [ ]:
#from pprint import pprint
for iter in range(5):
  elm_inputid = lm_datasets["train"][iter]["input_ids"]
  if(-100 not in elm_inputid):
    print(elm_inputid)
    print("Padded", len(elm_inputid))

  print("~"*100)
  elm_mask =lm_datasets["train"][iter]["attention_mask"]
  if(0 not in elm_mask):
    print(elm_mask)
    print("complete_masked", len(elm_mask))
  print("-"*270)

[796, 569, 18354, 7496, 17740, 6711, 796, 220, 198, 2311, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791, 47398, 17740, 357, 4960, 1058, 10545, 230, 99, 161, 254, 112, 5641, 44444, 9202, 25084, 24440, 12675, 11839, 18, 837, 6578, 764, 569, 18354, 7496, 286, 262, 30193, 513, 1267, 837, 8811, 6412, 284, 355, 569, 18354, 7496, 17740, 6711, 2354, 2869, 837, 318, 257, 16106, 2597, 2488, 12, 31, 2712, 2008, 983, 4166, 416, 29490, 290, 6343, 13, 44206, 329, 262, 14047, 44685, 764, 28728, 287, 3269, 2813, 287, 2869, 837, 340, 318, 262, 2368, 983, 287, 262, 569, 18354, 7496, 2168, 764, 12645, 278, 262, 976, 21748, 286, 16106, 290, 1103, 2488, 12, 31, 640, 11327, 355, 663, 27677, 837, 262, 1621, 4539, 10730, 284, 262, 717]
Padded 128
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [ ]:
print(len(lm_datasets["train"]), len(lm_datasets["validation"]))

18665 1928


In [ ]:
lm_datasets.keys(), lm_datasets["train"][1].keys()

(dict_keys(['test', 'train', 'validation']),
 dict_keys(['input_ids', 'attention_mask', 'labels']))

Now that the data has been cleaned, we're ready to instantiate our `Trainer`. We will a model:

In [ ]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Training approaches:-(1)using .trainer() method, (2) accelerator loop

**Loss Selection**
Choose Internal Model Loss (AutoMaskedLM / Forward Pass Loss): Let the AutoModelForMaskedLM compute the cross-entropy loss internally by passing labels=input_ids (shifted internally by the model).

**Why:** Computing loss inside the model ensures proper handling of shifted next-token prediction targets and keeps mixed-precision scaling stable under Accelerate. Manual cross-entropy calculation outside the model requires flattening logits and shifting tokens manually, risking shape mismatch errors or full-precision bottlenecks.

More specifically:-
1)**"Cross entropy**" will calculate the loss of predicting Token N given Token N, which ruins the causal constraint, while "ForCausalLMLoss"
fixes this by shifting the logits and labels so that the logit at position N is compared against the actual token label at position N+1.

2)**"Standard Cross Entropy"**: Requires you to manually generate attention masks and multiply tensors to exclude the prompt or padding tokens
from the gradient calculation.Whrereas "ForCausalLMLoss Wrapper": Leverages PyTorch's default ignore_index=-100 convention. Any token in
your labels configuration set to -100 is automatically ignored during the cross-entropy computation, making masking user prompts incredibly clean.

**Note**:-"ForCausalLMLoss" is simply a specific wrapper implementation that calculates standard Cross-Entropy Loss while automatically handling token shifting and sequence padding).

*1)Trainer method*

In [ ]:
from transformers import DataCollatorForLanguageModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)# since causal language model(CLM) not masked language model(MLM)

To see how the random masking works, let’s feed a few examples to the data collator. Since it expects a list of dicts, where each dict represents a single chunk of contiguous text, we first iterate over the dataset before feeding the batch to the collator. We remove the "word_ids" key for this data collator as it does not expect it:

In [ ]:
samples = [lm_datasets["train"][i] for i in range(2)]
#for sample in samples:
 # _ = sample.pop("word_ids")

print(data_collator(samples))
print(type(data_collator(samples)))# output analogous to dictionary of list where list prepared by concatanation of similar key's values of elemental_dict of input list
#that data_collator requires(although output class displayed as <class 'transformers.tokenization_utils_base.BatchEncoding'>). Also elemental list when sliced to ith index
#gives same dictionary of list values of ith elemnts for given key(unique prop of <class 'datasets.arrow_dataset.Dataset'> of dict elements)

#for chunk in data_collator(samples):
  #print(chunk)
  #id_chunk = chunk["input_ids"]
  #print(f"id_chunk :{id_chunk}")
  #label_chunk = chunk["labels"]
  #print(f"label_chunk :{label_chunk}")
  #print(f"\n'>>> {tokenizer.decode(id_chunk)}'")
  #print("_"*100)

{'input_ids': tensor([[  796,   569, 18354,  7496, 17740,  6711,   796,   220,   198,  2311,
            73, 13090,   645,   569, 18354,  7496,   513,  1058,   791, 47398,
         17740,   357,  4960,  1058, 10545,   230,    99,   161,   254,   112,
          5641, 44444,  9202, 25084, 24440, 12675, 11839,    18,   837,  6578,
           764,   569, 18354,  7496,   286,   262, 30193,   513,  1267,   837,
          8811,  6412,   284,   355,   569, 18354,  7496, 17740,  6711,  2354,
          2869,   837,   318,   257, 16106,  2597,  2488,    12,    31,  2712,
          2008,   983,  4166,   416, 29490,   290,  6343,    13, 44206,   329,
           262, 14047, 44685,   764, 28728,   287,  3269,  2813,   287,  2869,
           837,   340,   318,   262,  2368,   983,   287,   262,   569, 18354,
          7496,  2168,   764, 12645,   278,   262,   976, 21748,   286, 16106,
           290,  1103,  2488,    12,    31,   640, 11327,   355,   663, 27677,
           837,   262,  1621,  4539, 1

And some `TrainingArguments`:

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
model_name = model_checkpoint.split("/")[-1]
training_args = TrainingArguments(
    output_dir= "./local_checkpoints"+f"{model_name}-finetuned-wikitext2",#avoid giving mounted drive path as models
    #will consumeheavy write operations that quickly max out your Drive storage and API limits
    eval_strategy = "epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    fp16=True,
    save_strategy="epoch",
    #save_steps=1000,          # Don't save too frequently, but to be used when both eval strategy and save strategy are "steps"
    save_total_limit=1,      # Automatically deletes older checkpoints; keeps only the newest
    load_best_model_at_end=True # Keeps your optimal weights
)

We pass along all of those to the `Trainer` class:

In [ ]:
'''
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
)
'''

'\ntrainer = Trainer(\n    model=model,\n    args=training_args,\n    train_dataset=lm_datasets["train"],\n    eval_dataset=lm_datasets["validation"],\n)\n'

The "tokenizer" argument in Hugging Face Trainer is deprecated, and processing_class is its modern replacement. processing_class accepts a tokenizer, image processor, feature extractor, or multimodal processor, allowing the Trainer to handle text, vision, and audio data uniformly.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator= data_collator,
    processing_class= tokenizer,#The "tokenizer" argument in Hugging Face Trainer is deprecated, and processing_class is its modern replacement.
)


'\nby default compute loss function for causal models is "ForCausalLMLoss". We should add arguments "compute_loss_func" as "cross entropy"(although\n"ForCausalLMLoss" is simply a specific wrapper implementation that calculates standard Cross-Entropy Loss while automatically handling token shifting and sequence padding).\nMore specifically:-\n1)"Cross entropy" will calculate the loss of predicting Token N given Token N, which ruins the causal constraint, while "ForCausalLMLoss"\nfixes this by shifting the logits and labels so that the logit at position N is compared against the actual token label at position N+1.\n2)"Standard Cross Entropy": Requires you to manually generate attention masks and multiply tensors to exclude the prompt or padding tokens\nfrom the gradient calculation.Whrereas "ForCausalLMLoss Wrapper": Leverages PyTorch\'s default ignore_index=-100 convention. Any token in\nyour labels configuration set to -100 is automatically ignored during the cross-entropy computation,

And we can train our model:

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,3.593756,3.614224
2,3.505880,3.606909
3,3.461849,3.606503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=7002, training_loss=3.5301306872189437, metrics={'train_runtime': 1594.8584, 'train_samples_per_second': 35.11, 'train_steps_per_second': 4.39, 'total_flos': 1828913943674880.0, 'train_loss': 3.5301306872189437, 'epoch': 3.0})

Once the training is completed, we can evaluate our model and get its perplexity on the validation set like this:

In [ ]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Training Loss,Validation Loss,Epoch
3.461849,3.606503,3


Perplexity: 36.84


*2) accelerator method*

If your preprocessed dataset already contains input_ids, attention_mask, and labels (which you can easily create via a .map() function by duplicating input_ids), using default_data_collator is perfectly fine and often preferred for static, equal-length training blocks.

DataCollatorForLanguageModeling(mlm=False) usually preferred where input ids of variable lengths, and label fields missing(creates padding tokens for batch maximum, assigns "PAD" tokens as -100 and create labels field on fly while training)

In [ ]:
#import torch
#import torchvision
from torch.utils.data import DataLoader
#from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator
from accelerate import Accelerator
from torch.optim import AdamW

tokenizer(AutoTokenizer), model(AutoModelForCausalLM) already initialized earlier. Also pre-processing(tokenization, concatanation, equal block prepaartion, label=input_ids duplication batch creation using map method with group_text function already done)

In [ ]:
print(lm_datasets["train"][1])
print(type(lm_datasets["train"]))

{'input_ids': [983, 290, 5679, 262, 366, 17871, 5321, 366, 837, 257, 23634, 2422, 4326, 7351, 262, 3277, 286, 7096, 544, 1141, 262, 5498, 1898, 6839, 1810, 508, 1620, 3200, 2042, 4560, 290, 389, 46852, 1028, 262, 11773, 4326, 366, 2199, 321, 265, 88, 12552, 366, 764, 220, 198, 383, 983, 2540, 2478, 287, 3050, 837, 6872, 625, 257, 1588, 6903, 286, 262, 670, 1760, 319, 569, 18354, 7496, 17740, 2873, 764, 2893, 340, 17383, 262, 3210, 3033, 286, 262, 2168, 837, 340, 635, 25289, 3294, 16895, 837, 884, 355, 1642, 262, 983, 517, 43486, 329, 2168, 29661, 764, 15684, 11915, 371, 4548, 64, 8835, 73, 280, 290, 26777, 7286, 13704, 13231, 43354, 1111, 4504, 422, 2180, 12784, 837, 1863, 351, 569, 18354, 7496, 17740, 2873, 3437, 33687, 5303, 18024], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
from torchcodec.decoders import VideoDecoder
import sys
import torchvision.io

# Create a dummy object to satisfy the internal import check
class DummyVideoReader:
    pass

torchvision.io.VideoReader = DummyVideoReader
sys.modules['torchvision.io'].VideoReader = DummyVideoReader


above codeblock to avoid "ImportError: cannot import name 'VideoReader' from 'torchvision.io' when using datasets with torch format" while iterating pytorch tensors(converted by .set_format(type="torch", columns=column_list of train/eval dataset)) or dataloader iteration during training loop

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

# Format both datasets to return PyTorch tensors
lm_datasets["train"].set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
lm_datasets["validation"].set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

batch_size = 32
train_dataloader = DataLoader(
    lm_datasets["train"],
    shuffle=True, #shuffles batches within without disturbing their own block permuation
    batch_size=batch_size,# .map() method in "group_text" and "tokenize_function" functions used batched=True, batch_size=1000.
     #But batch in map is purely for processing convenience and performance, not for grouping final output data like a PyTorch DataLoader
    collate_fn=default_data_collator,
)
eval_dataloader = DataLoader(
    lm_datasets["validation"],
    shuffle =False, #can be commented also
    batch_size=batch_size, #for very same reasons as in train_dataloader
    collate_fn=default_data_collator
)

In [ ]:
print(len(lm_datasets["train"]), len(train_dataloader))
print(len(lm_datasets["validation"]), len(eval_dataloader))

print(type(train_dataloader), type(lm_datasets["train"]))

18665 584
1928 61
<class 'torch.utils.data.dataloader.DataLoader'> <class 'datasets.arrow_dataset.Dataset'>


In [ ]:
#print(lm_datasets["train"][1])
#print(train_dataloader[0])

In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=3e-5,
    eps=1e-8,
)

In [ ]:
accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:

from transformers import get_linear_schedule_with_warmup

num_train_epochs = 3
#max_grad_norm = 1.0

# Total number of training steps is number of batches * number of epochs.
total_steps = len(train_dataloader) * num_train_epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [ ]:
#!pip install av
#!pip install torchcodec

In [ ]:
from tqdm.auto import tqdm
import torch
import math

progress_bar = tqdm(range(total_steps))

train_loss_values, eval_loss_values = [], []

for epoch in range(num_train_epochs):
    # Put the model into training mode.
    model.train()

    # Reset the total loss for this epoch.
    total_loss = 0

    for batch in train_dataloader:
        outputs = model(**batch)

        # get the loss
        loss = outputs.loss

        # Perform a backward pass to calculate the gradients
        accelerator.backward(loss)

        # track train loss
        total_loss+= loss.item()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Calculate the average loss over the training data.
    avg_train_loss = total_loss / len(train_dataloader)
    print(f">>> Epoch {epoch}: avg_Training/train Loss: {avg_train_loss}")

    # Store the loss value for plotting the learning curve.
    train_loss_values.append(avg_train_loss)

    # Evaluation
    model.eval()
    losses = []
    for step, batch in enumerate(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        loss = outputs.loss
        losses.append(accelerator.gather(loss.repeat(batch_size)))

    losses = torch.cat(losses)
    losses = losses[: len(eval_dataloader)]

    eval_loss = losses.mean().item()
    print(f">>> Epoch {epoch}: avg_Validation/eval Loss: {eval_loss}")

    # Store the loss value for plotting the learning curve.
    eval_loss_values.append(losses.mean().item())

    try:
        perplexity = math.exp(torch.mean(losses))
    except OverflowError:
        perplexity = float("inf")

    print(f">>> Epoch {epoch}: Perplexity: {perplexity}")

  0%|          | 0/1752 [00:00<?, ?it/s]

>>> Epoch 0: avg_Training/train Loss: 3.490426443619271
>>> Epoch 0: avg_Validation/eval Loss: 3.551253318786621
>>> Epoch 0: Perplexity: 34.85697702637886
>>> Epoch 1: avg_Training/train Loss: 3.438581409111415
>>> Epoch 1: avg_Validation/eval Loss: 3.549760580062866
>>> Epoch 1: Perplexity: 34.804983483020955
>>> Epoch 2: avg_Training/train Loss: 3.4099241021561295
>>> Epoch 2: avg_Validation/eval Loss: 3.5482380390167236
>>> Epoch 2: Perplexity: 34.75203178785151


# Use save method of whichever approach gives lower perplexity

**push to Git/save by trainer method**

You can now upload the result of the training to the Hub, just execute this instruction:

In [ ]:
#trainer.push_to_hub()

In [ ]:
trainer.save_model(path+ f"{model_name}-finetuned-wikitext2")# also saves training_args.bin(not saved in accelerator method below)

**save by acclereate loop method**

In [ ]:
# 1. Wait for all processes to finish training
accelerator.wait_for_everyone()

# 2. Unwrap the model to strip away the distributed training shell
unwrapped_model = accelerator.unwrap_model(model)#Unwrap Model: accelerator.unwrap_model(model) removes the extra layers added by
#accelerator.prepare() so you get your raw base model back.

# 3. Save only on the main process to avoid file conflicts
if accelerator.is_main_process:
    # This creates the weights and the required config.json file

    model_name = model_checkpoint.split("/")[-1]
    unwrapped_model.save_pretrained(path+ f"{model_name}-finetuned-wikitext2", save_function=accelerator.save)#Save Function: save_function=accelerator.save makes
    #sure the model saves correctly across multiple GPUs or machines without breaking
    tokenizer.save_pretrained(path+ f"{model_name}-finetuned-wikitext2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Load saved model for inference

You can now share this model with all your friends, family, favorite pets: they can all load it with the identifier `"your-username/the-name-you-picked"` so for instance:

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("sgugger/my-awesome-model")
```

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

path ="/content/gdrive/MyDrive/"

model_checkpoint = "distilgpt2"
#model_checkpoint ="distilbert/distilgpt2"
model_name = model_checkpoint.split("/")[-1]

tokenizer_infer =AutoTokenizer.from_pretrained(path+ f"{model_name}-finetuned-wikitext2")
model_infer = AutoModelForCausalLM.from_pretrained(path+ f"{model_name}-finetuned-wikitext2")

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

In [ ]:
print(len(lm_datasets["test"]),len(lm_datasets["test"][1]["input_ids"]), len(lm_datasets["test"][1]["labels"]), type(lm_datasets["test"][1]["labels"]))
print(tokenizer.decode(lm_datasets["test"][1]["input_ids"]))
print(tokenizer.decode(lm_datasets["test"][1]["labels"]))
print("-"*60)
print(tokenizer_infer.decode(lm_datasets["test"][1]["input_ids"]))
print(tokenizer_infer.decode(lm_datasets["test"][1]["labels"]))
print("@"*60)


2210 128 128 <class 'list'>
 the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the Drum Theatre in Plymouth and the Menier Chocolate Factory in London . He was directed by John Tiffany and starred alongside Ben Whishaw , Shane Zaza , Harry Kent , Fraser Ayres , Sophie Stanton and Dominic Hall . 
 In 2006 , Boulter starred alongside Whishaw in the play Citizenship written by Mark Ravenhill . He appeared on a 2006 episode of the television series , Doctors , followed by a role in the 2007 theatre production of How to Curse directed by Josie Rourke . How to Curse was performed at Bush Theatre in the London
 the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was performed at the Drum Theatre in Plymouth and the Menier Chocolate Factory in London . He was directed by John Tiffany and starred alongside Ben Whishaw , Shane Zaza , Harry Kent , Fraser Ayres , Sophie Stanton and Dominic Hall . 
 In 2006 , Boulter starred alongside

# "optimizing number of next tokens"(Decoding/sampling strategies)


(topk &/or top_p) sampling together, used where diversity and creativity of response is important. Robust for non-repeatations on higher lengths.

In [ ]:


prompt = "the 2005 theatre productions of the Philip Ridley play Mercury Fur ,"

#inputs = tokenizer_infer(prompt, return_tensors="pt")
inputs = tokenizer_infer(prompt, return_tensors="pt").input_ids
print(type(inputs))
print(inputs)
outputs_sample_k_p = model_infer.generate(inputs, max_new_tokens=25, do_sample=True, top_k=20, top_p=0.95)#every iter returns new o/p, since sampling
print(tokenizer_infer.batch_decode(outputs_sample_k_p, skip_special_tokens=True))

<class 'torch.Tensor'>
tensor([[ 1169,  5075, 21421, 32260,   286,   262, 14576, 39616,   711, 21673,
         22384,   837]])
["the 2005 theatre productions of the Philip Ridley play Mercury Fur , which is the only non @-@ production directed by Ridley 's son , Sir John Ridley . The film also has a"]


Greedy search is the default decoding strategy. Unless specified in GenerationConfig, this strategy generates a maximum of 20 new tokens.Used where factual accurcy and deterministic output/reponse is pertinent. Prone to repeatinons on high token length

In [ ]:
inputs = tokenizer_infer(prompt, return_tensors="pt")#not .input_ids method as for sampling
outputs_greedy = model_infer.generate(**inputs, max_new_tokens=20)
print(tokenizer_infer.batch_decode(outputs_greedy, skip_special_tokens=True))# here every iter returns same o/p as topmost probablity sampled for successive samples

['the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was nominated for the Academy Award for Best Actor in a Comedy Series . \n = = =']


**Beam search** keeps track of several generated sequences (beams) at each time step. After a certain number of steps, it selects the sequence with the highest overall probability. Unlike greedy search, this strategy can “look ahead” and pick a sequence with a higher probability overall even if the initial tokens have a lower probability.

You can also use do_sample=True with beam search to sample at each step, but beam search will still greedily prune out low probability sequences between steps.

best suited for input-grounded tasks, like describing an image or speech recognition.

1st code block for standard beam search, 2nd for multinomial(beam serach+ sampling)


In [ ]:
inputs = tokenizer_infer(prompt, return_tensors="pt")#not .input_ids method as for sampling, same as in greedy; here complete output dict(keys:-keys (input_ids, attention_mask) needed not only tensors(as .input_ids genreate)
outputs_beam_search = model_infer.generate(**inputs, max_new_tokens=20, num_beams =2)#if num_beams =1, its same as greedy i.e same o/p, but as num beams farther(3,4,5 etc); o/p not exactly same but still similar
print(tokenizer_infer.batch_decode(outputs_beam_search, skip_special_tokens=True))#same/highly similar o/p if non sampled in every iterations

['the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was nominated for the Academy Award for Best Actor in a Comedy Series . \n = = =']


In [ ]:
inputs = tokenizer_infer(prompt, return_tensors="pt")#not .input_ids method as for sampling, same as in greedy; here complete output dict(keys:-keys (input_ids, attention_mask) needed not only tensors(as .input_ids genreate)
outputs_beam_search2 = model_infer.generate(**inputs, max_new_tokens=20, do_sample=True, num_beams=2)
print(tokenizer_infer.batch_decode(outputs_beam_search2, skip_special_tokens=True)) #each iter again new o/p, since sampling

['the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was also included on the list of " the most successful " productions of the 20th century .']


**Effect of tempearture**:- only applicale where "do_sample =True"(by default its false as in pure greedy or pure/standard beam search. This means tempreature only applicable in sampling case be it top k, top p, or multinomal(combination of "do_sample" =True and "num_beams">=1))

Lower temperature(T< 1) sharpens distribution making choices predictable(more like greedy or standard beam), while higher tempearture(T> 1) flattens distribution to make diverse o/ps(greater randomness than normal top_p or top_k head samplings)

In [ ]:
inputs = tokenizer_infer(prompt, return_tensors="pt").input_ids
outputs_sample_k_p_temp = model_infer.generate(inputs, max_new_tokens=25, temperature =0.7,do_sample=True, top_k=20, top_p=0.95)#every iter returns new o/p, since sampling
print(tokenizer_infer.batch_decode(outputs_sample_k_p_temp, skip_special_tokens=True))#every new iter, diff results as T far from 0

['the 2005 theatre productions of the Philip Ridley play Mercury Fur , and the 2001 Shakespeare play The Tempest . The company also produced the musical The Tempest , and produced the musical The Tempest . The']


In [ ]:
inputs = tokenizer_infer(prompt, return_tensors="pt").input_ids
outputs_sample_k_p_temp_near0 = model_infer.generate(inputs, max_new_tokens=25, temperature =0.1,do_sample=True, top_k=20, top_p=0.95)#every iter returns new o/p, since sampling
print(tokenizer_infer.batch_decode(outputs_sample_k_p_temp_near0, skip_special_tokens=True))# but quite similar respone/o-p with each iter as T near to 0[note o/p similarity with greedy or standard beam]

['the 2005 theatre productions of the Philip Ridley play Mercury Fur , which was nominated for the Academy Award for Best Actor in a Comedy Series . \n = = = Awards and nominations = =']


# **===============================================================================**